# Normalization-grid diagnostics

This notebook visualizes the deterministic Dalitz grid used for normalization in `DalitzPlotFitter`.

The current `DalitzGrid` is **not** a rectangular grid followed by a physical-region rejection. Instead, a regular midpoint grid in auxiliary coordinates \(u,v\)\in[0,1]^2\) is mapped directly into the physical \(s_{12},s_{13}\) Dalitz region:

- \(u\) divides the physical Dalitz area into equal-area strips in \(s_{12}\);
- \(v\) divides the allowed \(s_{13}\) interval at fixed \(s_{12}\) into equal subdivisions;
- all \(N^2\) points are physical;
- all quadrature weights are identical and equal to the total numerical Dalitz area.

This makes the normalization deterministic and removes Monte Carlo fluctuations from the normalization integral.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import DalitzGrid, DecayChannel, enable_x64
from dalitzplotfitter.kinematics import dalitz_s13_limits

enable_x64()


## 1. Define the decay channel and normalization grid

The default example is \(D^+\to\pi^-\pi^+\pi^+\), matching the other example notebooks.

`GRID_N` is the actual normalization-grid resolution. With `GRID_N = 800`, the grid contains \(800^2=640000\) physical points.


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

GRID_N = 800

grid = DalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    resolution=GRID_N,
)

norm_grid = grid.sample()

print(f"channel              : {channel.parent} -> {' '.join(channel.daughters)}")
print(f"grid resolution      : {GRID_N} x {GRID_N}")
print(f"number of grid points: {norm_grid.size:,}")
print(f"Dalitz area          : {float(grid.area):.12f} GeV^4")
print(f"minimum weight       : {float(jnp.min(norm_grid.weights)):.12f}")
print(f"maximum weight       : {float(jnp.max(norm_grid.weights)):.12f}")
print(f"weight spread        : {float(jnp.ptp(norm_grid.weights)):.3e}")


## 2. Physical Dalitz boundary and all normalization points

The black curve is the exact kinematic boundary obtained from `dalitz_s13_limits`.

For the production-size \(800\times800\) grid there are 640k points. Matplotlib can draw all of them, but a tiny marker and rasterization are used to keep the figure responsive.


In [ ]:
m1, m2, m3 = channel.daughter_masses
M = channel.parent_mass

s12_min = (m1 + m2)**2
s12_max = (M - m3)**2

s12_boundary = jnp.linspace(s12_min, s12_max, 2000)
s13_low, s13_high = dalitz_s13_limits(
    s12_boundary,
    mother_mass=M,
    masses=channel.daughter_masses,
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    np.asarray(norm_grid.s12),
    np.asarray(norm_grid.s13),
    s=0.08,
    alpha=0.45,
    rasterized=True,
    label=f"{GRID_N} x {GRID_N} normalization grid",
)

ax.plot(np.asarray(s12_boundary), np.asarray(s13_low), linewidth=1.4)
ax.plot(np.asarray(s12_boundary), np.asarray(s13_high), linewidth=1.4)

ax.set_xlabel(r"$s_{12}=m^2(1,2)\;[\mathrm{GeV}^2]$")
ax.set_ylabel(r"$s_{13}=m^2(1,3)\;[\mathrm{GeV}^2]$")
ax.set_title("Physical Dalitz region and normalization grid")
ax.legend(markerscale=20)
ax.set_aspect("equal", adjustable="box")
plt.show()


## 3. Make the grid structure visible

At full resolution the grid looks almost continuous. The next plot draws only selected \(u\)-strips and selected \(v\)-columns from the **same normalization grid**.

This shows an important property of the mapping: spacing is regular in the auxiliary equal-area coordinates, not in \(s_{12}\) and \(s_{13}\).


In [ ]:
s12_matrix = np.asarray(norm_grid.s12).reshape(GRID_N, GRID_N)
s13_matrix = np.asarray(norm_grid.s13).reshape(GRID_N, GRID_N)

# Select a manageable number of rows/columns only for visualization.
n_lines = 35
row_idx = np.unique(np.linspace(0, GRID_N - 1, n_lines, dtype=int))
col_idx = np.unique(np.linspace(0, GRID_N - 1, n_lines, dtype=int))

fig, ax = plt.subplots(figsize=(8, 7))

for i in row_idx:
    ax.plot(s12_matrix[i, :], s13_matrix[i, :], linewidth=0.55, alpha=0.75)

for j in col_idx:
    ax.plot(s12_matrix[:, j], s13_matrix[:, j], linewidth=0.55, alpha=0.75)

ax.plot(np.asarray(s12_boundary), np.asarray(s13_low), linewidth=1.5)
ax.plot(np.asarray(s12_boundary), np.asarray(s13_high), linewidth=1.5)

ax.set_xlabel(r"$s_{12}\;[\mathrm{GeV}^2]$")
ax.set_ylabel(r"$s_{13}\;[\mathrm{GeV}^2]$")
ax.set_title("Selected coordinate lines from the normalization grid")
ax.set_aspect("equal", adjustable="box")
plt.show()


## 4. Equal-area strips in \(s_{12}\)

The \(s_{12}\) coordinates are intentionally **not equally spaced**.

Each row of the \(N\times N\) grid corresponds to one equal-area strip. Near regions where the physical \(s_{13}\) width is small, the strips become wider in \(s_{12}\); where the Dalitz region is broad, they become narrower.


In [ ]:
s12_strip = s12_matrix[:, 0]
delta_s12 = np.diff(s12_strip)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(delta_s12.size), delta_s12)
ax.set_xlabel("equal-area strip index")
ax.set_ylabel(r"$\Delta s_{12}$ between strip midpoints $[\mathrm{GeV}^2]$")
ax.set_title(r"Non-uniform $s_{12}$ spacing required for equal physical area")
plt.show()

print(f"min Δs12 = {delta_s12.min():.6e} GeV^2")
print(f"max Δs12 = {delta_s12.max():.6e} GeV^2")
print(f"max/min   = {delta_s12.max()/delta_s12.min():.3f}")


## 5. The auxiliary \(u,v\) midpoint grid

Before mapping to Dalitz coordinates, the construction is simply a regular midpoint grid:

\[
u_i = \frac{i+1/2}{N},\qquad
v_j = \frac{j+1/2}{N}.
\]

The non-trivial geometry appears only after mapping \(u,v\)\rightarrow(s_{12},s_{13}\).


In [ ]:
# A small auxiliary grid is enough to show the topology.
N_AUX_SHOW = 18
u_show = (np.arange(N_AUX_SHOW) + 0.5) / N_AUX_SHOW
v_show = (np.arange(N_AUX_SHOW) + 0.5) / N_AUX_SHOW
uu, vv = np.meshgrid(u_show, v_show, indexing="ij")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(uu.ravel(), vv.ravel(), s=12)
ax.set_xlabel(r"$u$")
ax.set_ylabel(r"$v$")
ax.set_title(r"Regular midpoint grid in auxiliary $(u,v)$ coordinates")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal", adjustable="box")
plt.show()


## 6. Quadrature weights

Because the transformation has constant Jacobian equal to the total Dalitz area, every grid point has the same stored weight.

Under the package convention

`mean(weights * f)`

the integral of a constant function \(f=1\) must reproduce the numerical area of the Dalitz region.


In [ ]:
weights = np.asarray(norm_grid.weights)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(weights, bins=30)
ax.set_xlabel("stored quadrature weight")
ax.set_ylabel("number of grid points")
ax.set_title("Normalization-grid weights")
plt.show()

integral_of_one = float(jnp.mean(norm_grid.weights * jnp.ones_like(norm_grid.weights)))

print(f"grid.area              = {float(grid.area):.12f}")
print(f"integral of f=1        = {integral_of_one:.12f}")
print(f"absolute difference    = {abs(integral_of_one - float(grid.area)):.3e}")
print(f"relative difference    = {abs(integral_of_one/float(grid.area) - 1.0):.3e}")


## 7. Check that every point is inside the exact physical boundary

This is a useful numerical sanity check. Unlike a rectangular midpoint grid with a mask, `DalitzGrid.sample()` should contain **zero rejected/outside points** by construction.


In [ ]:
low_at_grid, high_at_grid = dalitz_s13_limits(
    norm_grid.s12,
    mother_mass=M,
    masses=channel.daughter_masses,
)

tol = 1e-12
inside = (norm_grid.s13 >= low_at_grid - tol) & (norm_grid.s13 <= high_at_grid + tol)

n_inside = int(jnp.sum(inside))
n_outside = norm_grid.size - n_inside

print(f"inside points : {n_inside:,}")
print(f"outside points: {n_outside:,}")
print(f"inside fraction: {n_inside / norm_grid.size:.12f}")

assert n_outside == 0


## 8. Compare several resolutions

This final diagnostic makes it easy to inspect how the deterministic quadrature fills the same physical region as the resolution increases.

The physical boundary does not change; only the equal-area midpoint sampling becomes denser.


In [ ]:
resolutions = [10, 25, 60, 120]

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)

for ax, n in zip(axes.flat, resolutions):
    sample = DalitzGrid(
        M,
        channel.daughter_masses,
        resolution=n,
    ).sample()

    ax.scatter(np.asarray(sample.s12), np.asarray(sample.s13), s=5, alpha=0.7)
    ax.plot(np.asarray(s12_boundary), np.asarray(s13_low), linewidth=1.0)
    ax.plot(np.asarray(s12_boundary), np.asarray(s13_high), linewidth=1.0)
    ax.set_title(f"{n} x {n} = {sample.size:,} points")
    ax.set_aspect("equal", adjustable="box")

for ax in axes[-1, :]:
    ax.set_xlabel(r"$s_{12}\;[\mathrm{GeV}^2]$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$s_{13}\;[\mathrm{GeV}^2]$")

fig.suptitle("Deterministic equal-area Dalitz grids", y=0.995)
fig.tight_layout()
plt.show()


## Interpretation

The main points to verify visually are:

1. the normalization points fill only the physical Dalitz region;
2. there is no rectangular-grid rejection step in the current implementation;
3. rows correspond to equal-area strips rather than equally spaced \(s_{12}\);
4. all points carry the same normalization weight;
5. increasing `GRID_N` only refines the deterministic quadrature.

For fits, the same object created here can be passed as the normalization sample:

```python
norm_grid = DalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    resolution=800,
).sample()

cache = model.prepare_cache(data, norm_grid)
```
